# Hybrid Search and GraphRAG

This notebook introduces **hybrid search** — combining vector similarity and fulltext keyword search in a single retriever. You'll use `HybridRetriever` to blend both search methods, then add graph traversal with `HybridCypherRetriever` to build the most capable retrieval pipeline in the lab.

**Learning Objectives:**
- Use `HybridRetriever` for combined vector and fulltext search
- Tune the `alpha` parameter to control the vector/fulltext balance
- Use `HybridCypherRetriever` to add graph traversal to hybrid search
- Build a full GraphRAG pipeline with hybrid retrieval

**Key concepts:**
- `HybridRetriever` runs both vector and fulltext indexes simultaneously, normalizes scores, and ranks results
- `alpha=1.0` = pure vector (semantic), `alpha=0.0` = pure fulltext (keyword), `alpha=0.5` = balanced
- `HybridCypherRetriever` adds Cypher-based graph traversal on top of hybrid search

**Prerequisites:** Run notebooks 01 (data loading) and 02 (embeddings) first to create the graph and indexes.

In [ ]:
%pip install "neo4j-graphrag[bedrock] @ git+https://github.com/neo4j-partners/neo4j-graphrag-python.git@bedrock-embeddings" -q

In [ ]:
import os

from neo4j import GraphDatabase
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.retrievers import HybridRetriever, HybridCypherRetriever
from dotenv import load_dotenv

from lib.data_utils import get_embedder, get_llm

# Load configuration
load_dotenv('../CONFIG.txt')

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()

embedder = get_embedder()
llm = get_llm()

print(f'Connected to Neo4j!')
print(f'Embedder: {embedder.model_id}')
print(f'LLM: {llm.model_id}')

## Check Indexes

Hybrid search requires both a vector index and a fulltext index on the same node type. Verify both exist before proceeding.

In [ ]:
VECTOR_INDEX = 'chunkEmbeddings'
FULLTEXT_INDEX = 'search_chunks'

with driver.session() as session:
    result = session.run(
        "SHOW FULLTEXT INDEXES YIELD name WHERE name = $name RETURN name",
        name=FULLTEXT_INDEX,
    )
    assert result.single(), f"Fulltext index '{FULLTEXT_INDEX}' not found. Run notebook 01 first."

    result = session.run(
        "SHOW VECTOR INDEXES YIELD name WHERE name = $name RETURN name",
        name=VECTOR_INDEX,
    )
    assert result.single(), f"Vector index '{VECTOR_INDEX}' not found. Run notebook 02 first."

print(f'Vector index: {VECTOR_INDEX} ✓')
print(f'Fulltext index: {FULLTEXT_INDEX} ✓')

## Hybrid Retriever

The `HybridRetriever` searches both the vector index and the fulltext index simultaneously, normalizes the scores, and merges the results. The combined score is computed as:

```
combined_score = alpha * vector_score + (1 - alpha) * fulltext_score
```

In [ ]:
hybrid_retriever = HybridRetriever(
    driver=driver,
    vector_index_name=VECTOR_INDEX,
    fulltext_index_name=FULLTEXT_INDEX,
    embedder=embedder,
    return_properties=['text'],
)

print('HybridRetriever initialized!')

## Compare Alpha Values

The query "Apple revenue growth" contains both a specific entity name (best matched by fulltext) and a semantic concept (best matched by vector). Varying alpha shows how each component contributes.

In [ ]:
query = 'Apple revenue growth'

for alpha in [1.0, 0.5, 0.0]:
    results = hybrid_retriever.search(
        query_text=query,
        top_k=3,
        alpha=alpha,
    )

    label = {1.0: 'Pure Vector', 0.5: 'Balanced', 0.0: 'Pure Fulltext'}[alpha]

    print(f'\n=== Alpha={alpha} ({label}) ===')
    for i, item in enumerate(results.items, 1):
        score = item.metadata.get('score', 0.0) if item.metadata else 0.0
        text = str(item.content)[:120]
        print(f'{i}. Score: {score:.4f} | {text}...')

## Hybrid with Graph Context

The `HybridCypherRetriever` adds graph traversal to hybrid search, just as `VectorCypherRetriever` added it to vector search. After hybrid search finds matching chunks, the retrieval query traverses `FROM_DOCUMENT` to the source document, then follows the `FILED` relationship to find the company that filed it — pulling in connected entity nodes like products and risk factors.

In [ ]:
RETRIEVAL_QUERY = """
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (doc)<-[:FILED]-(company:Company)
OPTIONAL MATCH (company)-[:FACES_RISK]->(risk:RiskFactor)
OPTIONAL MATCH (product:Product)-[:FROM_CHUNK]->(node)
WITH node, doc, score,
     collect(DISTINCT company.name) AS companies,
     collect(DISTINCT risk.name)[0..5] AS risks,
     collect(DISTINCT product.name)[0..5] AS products
RETURN node.text AS text,
       score,
       {document: doc.name, companies: companies, products: products, risks: risks} AS metadata
"""

hybrid_cypher_retriever = HybridCypherRetriever(
    driver=driver,
    vector_index_name=VECTOR_INDEX,
    fulltext_index_name=FULLTEXT_INDEX,
    embedder=embedder,
    retrieval_query=RETRIEVAL_QUERY,
)

print('HybridCypherRetriever initialized!')

## Full GraphRAG Pipeline

Build the complete pipeline: hybrid search finds relevant chunks using both vector and keyword matching, graph traversal adds entity context (companies, products, risk factors), and the LLM generates an answer from the enriched results.

In [ ]:
rag = GraphRAG(llm=llm, retriever=hybrid_cypher_retriever)

query = "What are the key financial performance indicators and risk factors in Apple's filing?"
response = rag.search(query, retriever_config={'top_k': 5}, return_context=True)

print(f'Query: "{query}"\n')
print('Answer:')
print(response.answer)

print('\n\n=== Retrieved Context ===')
for i, item in enumerate(response.retriever_result.items, 1):
    content_str = str(item.content)
    preview = content_str[:250] + '...' if len(content_str) > 250 else content_str
    print(f'\n[{i}] {preview}')

## Retriever Selection Guide

You have now built four retriever configurations across this lab. Each fits a different query shape:

| Retriever | Search Method | Graph Context | Best For |
|-----------|--------------|---------------|----------|
| **VectorRetriever** | Vector similarity | No | General semantic questions |
| **VectorCypherRetriever** | Vector similarity | Companies, products, risk factors | Questions needing entity context from the knowledge graph |
| **HybridRetriever** | Vector + Fulltext | No | Queries mixing specific terms and semantic meaning |
| **HybridCypherRetriever** | Vector + Fulltext | Companies, products, risk factors | Queries needing both keyword precision and entity context |

**Alpha tuning guidelines:**
- Start with `alpha=0.5` for a balanced baseline
- Increase toward `1.0` for natural language questions without specific terms
- Decrease toward `0.0` for queries with entity names, tickers, or exact phrases

## Summary

This lab built a complete GraphRAG pipeline progressing through four retriever configurations:

1. **VectorRetriever** found chunks by semantic meaning
2. **VectorCypherRetriever** enriched matches with entity traversal
3. **HybridRetriever** added fulltext keyword precision to vector search
4. **HybridCypherRetriever** combined all three — vector, fulltext, and graph traversal

Each layer addresses a limitation of the previous one. Vector search misses exact terms; fulltext search misses semantic meaning; neither alone captures the entity relationships in the knowledge graph. The hybrid-cypher combination draws on all three.

---

**Next:** Continue to [Lab 6](../Lab_6_Advanced_Agents/) to build MCP agents that use these retrieval patterns as tools.

In [ ]:
# Cleanup
driver.close()
print('Connection closed.')